# 🏎️ Visualização de Velocidade no Circuito de F1

Este notebook analisa a telemetria real de um carro de Fórmula 1 para criar uma visualização do circuito colorida pela velocidade. Vamos usar dados da **qualificação do GP do Brasil de 2021**, mostrando a volta mais rápida de **Max Verstappen**.

## 1. Instalação das Dependências

Vamos instalar as bibliotecas necessárias. O FastF1 versão **< 3.9.0** é usada para evitar a autenticação do F1TV.

In [ ]:
!pip install "fastf1<3.9.0" matplotlib pandas numpy scipy

## 2. Importações e Configuração

Configuramos o cache para armazenar os dados baixados e acelerar carregamentos futuros.

In [ ]:
import os
import fastf1
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import numpy as np

# Cria pasta de cache se não existir
os.makedirs('f1_cache', exist_ok=True)

# Habilita o cache do FastF1
fastf1.Cache.enable_cache('f1_cache')

## 3. Carregando a Sessão

Carregamos a sessão de qualificação do GP do Brasil de 2021. O cache evitará baixar os dados novamente.

In [ ]:
# Carrega a sessão de qualificação
session = fastf1.get_session(2021, 'Brazil', 'Q')
session.load()

print("✅ Sessão carregada com sucesso!")
print(f"Total de voltas na sessão: {len(session.laps)}")

## 4. Extracting o Melhor Tempo

Vamos buscar a volta mais rápida de Max Verstappen (VER), campeão mundial de 2021 e 2022.

In [ ]:
# Escolhe o piloto e pega a volta mais rápida
piloto = 'VER'
volta_rapida = session.laps.pick_drivers(piloto).pick_fastest()

print(f"Piloto: {piloto}")
print(f"Tempo da volta: {volta_rapida['LapTime']}")
print(f"Composto do pneu: {volta_rapida['Compound']}")

## 5. Telemetria da Volta

A telemetria fornece dados de posição (X, Y) e velocidade a cada fração de segundo.

In [ ]:
# Extrai os dados de telemetria
telemetria = volta_rapida.get_telemetry()

print(f"Total de pontos de telemetria: {len(telemetria)}")
print(f"\nColunas disponíveis: {telemetria.columns.tolist()}")

## 6. Preparação dos Dados

Separar as coordenadas X, Y e a velocidade para a visualização.

In [ ]:
# Extrai as colunas necessárias
x = telemetria['X']
y = telemetria['Y']
speed = telemetria['Speed']

print(f"Velocidade mínima: {speed.min():.1f} km/h")
print(f"Velocidade máxima: {speed.max():.1f} km/h")
print(f"\nRange de coordenadas:")
print(f"X: de {x.min():.1f} até {x.max():.1f}")
print(f"Y: de {y.min():.1f} até {y.max():.1f}")

## 7. Visualização do Mapa de Velocidade

Criamos o gráfico colorido pelo mapa de calor. Cada segmento da pista recebe uma cor baseada na velocidade média daquele trecho.

In [ ]:
# Cria os segmentos de linha para o LineCollection
pontos = np.array([x, y]).T.reshape(-1, 1, 2)
segmentos = np.concatenate([pontos[:-1], pontos[1:]], axis=1)

# Cria a figura
fig, ax = plt.subplots(figsize=(14, 12))
fig.patch.set_facecolor('white')

# Cria a coleção de linhas coloridas por velocidade
lc = LineCollection(segmentos, cmap='RdYlGn_r', linewidth=6)
lc.set_array(speed)

# Adiciona ao gráfico
ax.add_collection(lc)
ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())
ax.set_aspect('equal')
ax.axis('off')

# Barra de cores
cbar = plt.colorbar(lc, ax=ax, pad=0.02)
cbar.set_label('Velocidade (km/h)', rotation=270, labelpad=20, fontsize=12)

# Títulos
plt.title(f'Mapa de Velocidade - Circuito de Interlagos', fontsize=18, fontweight='bold', pad=20)
plt.suptitle(f'{piloto} | GP do Brasil 2021 | Volta Mais Rápida', fontsize=14, y=0.93)

# Salva o gráfico
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/circuit_speed_map.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\n✅ Gráfico salvo em 'outputs/circuit_speed_map.png'")

## 8. Análise dos Resultados

A visualização revela as características do circuito de Interlagos:

- **Zonas vermelhas/yellow**: Trechos de alta velocidade (retas e curvas de médio raio)
- **Zonas verdes**: Trechos de baixa velocidade (frenagens e curvas lentas)

O traçado mostra claramente:
- A reta dos pitons (saída de curva 1)
- A curva 3 (Junção da Lagoa) - frenagem pesada
- A curva 4 (Sul) - alta velocidade
- A curva 5 (Boca do Boi) - curva lenta de 180°
- A reta dos mistérios (entre curvas 6 e 7)